# RoBERTa-base fine-tune — Name That LLM

Adapted from [`07_roberta_finetune.ipynb`](07_roberta_finetune.ipynb) for this competition:

| | Original notebook | This notebook |
| --- | --- | --- |
| Data | RAID parquet, 12 classes (incl. `human`, `gpt2`) | competition `train.csv`, **18** frontier models |
| Validation | random split | **prompt-grouped** fold (same `FOLD_SEED = 0` rule as the LoRA notebooks) |
| Metric | accuracy | **balanced accuracy** (competition metric) |
| Install | `pip install` | none — internet is off at submit |
| Output | `.pt` + plots | `submission.parquet` + saved probabilities for blending |

**Flow:** train on folds 1–4 and validate on fold 0 to pick the epoch count. Then retrain on all 2,699 rows for that many epochs and predict `test.csv`. Both steps run on one T4 in minutes, so this single notebook trains and submits.

**Expectation:** the original's 85–90% came from an easy dataset (human vs GPT-2 etc.). Here the classes are near-identical siblings (`gemini-3.7-flash` vs `gemini-3.8-flash`) with only ~150 passages each, so expect this to land **well below** the 0.489 LoRA. Its main use is as a cheap extra member for a blend.

## Attach on Kaggle
1. Competition **Name That LLM**
2. **RoBERTa-base** weights as a Kaggle Model or dataset (any folder with a `config.json` whose `model_type` is `roberta`)
3. Accelerator: **GPU T4** (one is enough)

## 1. Imports

In [ ]:
import gc
import glob
import json
import os
import time
import warnings
from pathlib import Path

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import balanced_accuracy_score, classification_report
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKING = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('.')
T0 = time.time()
print('device', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Config

`max_length` is 512 because the longest passages run ~420 tokens; batches are padded dynamically, so short passages stay cheap.

In [ ]:
SEED = 42
FOLD_SEED = 0          # same prompt-fold rule as Qwen3-8B-base-Lora.ipynb
VALID_FOLD = 0

CONFIG = {
    'max_length': 512,
    'batch_size': 16,
    'max_epochs': 8,
    'lr': 3e-5,
    'warmup_ratio': 0.06,
    'weight_decay': 0.01,
    'label_smoothing': 0.0,
    'max_grad_norm': 1.0,
}
BALANCE = True         # Sinkhorn class balancing on the full hidden test
USE_FP16 = torch.cuda.is_available()

torch.manual_seed(SEED)
np.random.seed(SEED)
print(CONFIG, 'fp16' if USE_FP16 else 'fp32')

## 3. Find data and RoBERTa weights

In [ ]:
def find_csv(name):
    for p in (Path('/kaggle/input/competitions/name-that-llm') / name,
              Path('/kaggle/input/name-that-llm') / name,
              Path('data') / name):
        if p.exists():
            return p
    hits = sorted(Path('/kaggle/input').rglob(name)) if Path('/kaggle/input').is_dir() else []
    if hits:
        return hits[0]
    raise FileNotFoundError(name)


def find_roberta():
    patterns = ['/kaggle/input/**/config.json'] if Path('/kaggle/input').is_dir() else []
    for pattern in patterns:
        for cfg in sorted(glob.glob(pattern, recursive=True)):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            if c.get('model_type') == 'roberta' and c.get('hidden_size') == 768 and c.get('num_hidden_layers') == 12:
                return os.path.dirname(cfg)
    return 'roberta-base'  # hub name; only works with internet on


MODEL_DIR = find_roberta()
train = pd.read_csv(find_csv('train.csv'))
test = pd.read_csv(find_csv('test.csv'))
test['text'] = test['text'].fillna('').astype(str)

LABELS = sorted(train.model.unique())
K = len(LABELS)
y = train.model.map({l: i for i, l in enumerate(LABELS)}).values

prompts = np.array(sorted(train.prompt_id.unique()))
np.random.RandomState(FOLD_SEED).shuffle(prompts)
fold = train.prompt_id.map({p: i % 5 for i, p in enumerate(prompts)}).values
fit_idx = np.where(fold != VALID_FOLD)[0]
val_idx = np.where(fold == VALID_FOLD)[0]

print('model', MODEL_DIR)
print('train', train.shape, 'test', test.shape, 'classes', K)
print('fit rows', len(fit_idx), 'valid rows', len(val_idx),
      'shared prompts', len(set(train.prompt_id[fit_idx]) & set(train.prompt_id[val_idx])))

## 4. Dataset and helpers

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
collate = DataCollatorWithPadding(tok)


class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tok(list(texts), truncation=True, max_length=CONFIG['max_length'])
        self.labels = labels

    def __len__(self):
        return len(self.enc['input_ids'])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = int(self.labels[i])
        return item


def loader(texts, labels=None, shuffle=False, bs=None):
    return DataLoader(TextDataset(texts, labels), batch_size=bs or CONFIG['batch_size'],
                      shuffle=shuffle, collate_fn=collate, num_workers=2)


@torch.no_grad()
def predict_proba(model, dl):
    model.eval()
    out = []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != 'labels'}
        with torch.autocast('cuda', dtype=torch.float16, enabled=USE_FP16):
            logits = model(**b).logits
        out.append(torch.softmax(logits.float(), -1).cpu().numpy())
    return np.concatenate(out)


def new_model_and_opt(n_batches, epochs):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=K).to(DEVICE)
    no_decay = ('bias', 'LayerNorm.weight')
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': CONFIG['weight_decay']},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=CONFIG['lr'], eps=1e-8)
    total = n_batches * epochs
    sch = get_cosine_schedule_with_warmup(opt, int(total * CONFIG['warmup_ratio']), total)
    return model, opt, sch


def train_epoch(model, opt, sch, scaler, dl, crit):
    model.train()
    running = 0.0
    for b in dl:
        labels = b.pop('labels').to(DEVICE)
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast('cuda', dtype=torch.float16, enabled=USE_FP16):
            logits = model(**b).logits
        loss = crit(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
        scaler.step(opt)
        scaler.update()
        sch.step()
        running += loss.item()
    return running / len(dl)

## 5. Validation run — pick the epoch count

Train on folds 1–4, score fold 0 with **balanced accuracy** after each epoch. The best epoch count is reused for the full-data fit. The schedule is set for `max_epochs`, so the chosen epoch is a point on that same schedule.

In [ ]:
crit = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
texts = train.text.astype(str).values

fit_dl = loader(texts[fit_idx], y[fit_idx], shuffle=True)
val_dl = loader(texts[val_idx], bs=CONFIG['batch_size'] * 2)

model, opt, sch = new_model_and_opt(len(fit_dl), CONFIG['max_epochs'])
scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)

history = []
best = dict(score=-1.0, epoch=0, proba=None)
for epoch in range(1, CONFIG['max_epochs'] + 1):
    loss = train_epoch(model, opt, sch, scaler, fit_dl, crit)
    proba = predict_proba(model, val_dl)
    score = balanced_accuracy_score(y[val_idx], proba.argmax(1))
    history.append(dict(epoch=epoch, train_loss=round(loss, 4), val_bal_acc=round(score, 4)))
    print(f'epoch {epoch}: train_loss {loss:.4f}  val balanced acc {score:.4f}  ({time.time() - T0:.0f}s)', flush=True)
    if score > best['score']:
        best = dict(score=score, epoch=epoch, proba=proba)

print(pd.DataFrame(history).to_string(index=False))
print(f"best epoch {best['epoch']}  val balanced acc {best['score']:.4f}")
print(classification_report(y[val_idx], best['proba'].argmax(1), target_names=LABELS, digits=3, zero_division=0))

np.savez_compressed(WORKING / 'roberta_val_fold0.npz',
                    ids=train.id.values[val_idx], proba=best['proba'], classes=np.array(LABELS))
del model, opt, sch, scaler
gc.collect()
torch.cuda.empty_cache()

## 6. Full-data fit and test prediction

Retrain from scratch on all 2,699 rows for the chosen number of epochs (schedule scaled to that length), then predict `test.csv`.

In [ ]:
EPOCHS = best['epoch']
full_dl = loader(texts, y, shuffle=True)
test_dl = loader(test.text.values, bs=CONFIG['batch_size'] * 2)

model, opt, sch = new_model_and_opt(len(full_dl), EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, opt, sch, scaler, full_dl, crit)
    print(f'full epoch {epoch}/{EPOCHS}: train_loss {loss:.4f} ({time.time() - T0:.0f}s)', flush=True)

P = predict_proba(model, test_dl)
np.savez_compressed(WORKING / 'roberta_test_proba.npz',
                    ids=test.id.astype(str).values, proba=P, classes=np.array(LABELS))
print('test proba', P.shape)

## 7. Balance and write `submission.parquet`

Sinkhorn balancing nudges each class toward ~1/18 of the predictions. It is skipped on the 3-row interactive `test.csv`.

In [ ]:
Q = P.copy()
if BALANCE and len(Q) >= 50:
    target = np.full(K, len(Q) / K)
    for _ in range(100):
        Q *= target / np.clip(Q.sum(0), 1e-12, None)
        Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)

sub = pd.DataFrame({'id': test['id'].astype(str), 'model': np.array(LABELS)[Q.argmax(1)]})
assert sub['id'].is_unique and set(sub['model']) <= set(LABELS)
sub.to_parquet('submission.parquet', index=False)

json.dump({'val_balanced_acc_fold0': round(best['score'], 4), 'epochs': EPOCHS,
           'history': history, 'config': CONFIG, 'model_dir': MODEL_DIR},
          open(WORKING / 'roberta_results.json', 'w'), indent=2)
print(sub['model'].value_counts().sort_index().to_string())
print(f'done in {time.time() - T0:.0f}s -> submission.parquet ({len(sub)} rows)')